<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/ADPs/Lecture_5/%D0%9F%D1%80%D0%B0%D0%BA%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B0%D1%8F_%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%B0_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Практическая работа № 5: Современные NLP-модели в психологии

## Введение

В лекции №5 мы познакомились с тем, как современные NLP-модели (BERT, GPT и их открытые аналоги 2025–2026 годов) могут применяться в клинической психологии и психиатрии. Мы разобрали:

- как слова превращаются в векторы (эмбеддинги) и что это даёт для понимания смысла,
- как работает архитектура трансформера и механизм внимания,
- в чём различие между BERT (понимание) и GPT (генерация),
- какие открытые модели доступны для работы с русскоязычными психологическими текстами,
- как извлекать симптомы, анализировать эмоции, искать похожие клинические случаи,
- какие этические риски сопровождают использование LLM в психологии.

Теперь вам предстоит применить эти знания на практике.

**Цель работы** — закрепить навыки работы с предобученными моделями:
- загрузка моделей с Hugging Face,
- анализ эмоций в тексте пациента,
- поиск семантически близких клинических случаев,
- извлечение симптомов,
- оценка суицидального риска,
- критическая оценка этических аспектов использования LLM.

---

## Подготовка рабочей среды

Перед началом работы установите необходимые библиотеки (в терминале или командной строке):




In [ ]:
!pip install transformers torch sentence-transformers scikit-learn pandas numpy



Для работы с моделями может потребоваться **около 2–3 ГБ свободного места** для загрузки моделей. Если у вас ограниченный интернет-трафик или медленное соединение, вы можете работать с примерами кода в теории, не запуская их локально, или использовать более лёгкие модели (например, `rubert-tiny` вместо `rubert-large`).

Импортируйте необходимые модули:




In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd


## Часть 1. Теоретические вопросы (для самопроверки)

Перед выполнением практических заданий письменно ответьте на следующие вопросы. Это поможет убедиться, что вы понимаете ключевые концепции.

1. Что такое векторное представление слов (word embedding)? Почему оно важно для понимания текста моделями?

2. В чём основное отличие трансформера от предыдущих архитектур (RNN, LSTM)? Почему это важно для понимания текстов пациентов?

3. Что такое механизм внимания (attention)? Приведите аналогию из психотерапии.

4. В чём разница между само-вниманием (self-attention) и обычным вниманием?

5. Что такое предобучение (pre-training) и дообучение (fine-tuning)? Проведите аналогию с образованием психолога.

6. В чём ключевое различие между BERT и GPT? Для каких психологических задач подходит каждая модель?

7. Назовите три открытые модели 2025–2026 годов для работы с русскоязычными психологическими текстами и их задачи.

8. Что такое RAG (Retrieval-Augmented Generation) и как он помогает бороться с галлюцинациями?

9. Какие этические риски возникают при использовании NLP-моделей в психологии? Как их минимизировать?

10. **Рефлексивный вопрос:** как вы видите баланс между автоматизацией (модели ИИ) и человеческим контролем в психиатрии? Что должно оставаться за врачом?



## Часть 2. Практические задания на Python

Все задания выполняйте в Jupyter Notebook или отдельном Python-скрипте. Код должен быть снабжён комментариями на русском языке. В конце каждого задания приводите краткий анализ результатов с точки зрения психолога.

---

### Задание 1. Семантический анализ: поиск соседей для клинических терминов

**Описание.** В этом задании вы познакомитесь с векторными эмбеддингами на практике. Вы загрузите предобученную модель Word2Vec (или используете Sentence Transformers) и найдёте слова, семантически близкие к клиническим терминам.

**Требуется:**

1. Загрузите русскоязычную модель Word2Vec (`word2vec-ruscorpora-300`) через библиотеку `gensim.downloader`.

2. Найдите 10 слов, наиболее близких по смыслу к слову **«депрессия»**. Выведите их с коэффициентами сходства.

3. Найдите 10 слов, наиболее близких к слову **«тревога»**. Сравните полученные списки. Есть ли пересечения? Что это говорит о семантической близости этих концептов?

4. Проверьте семантическое расстояние между парами слов:
   - «депрессия» и «тоска»
   - «депрессия» и «радость»
   - «тревога» и «страх»
   - «тревога» и «спокойствие»

5. Найдите «лишнее» слово в списке: `["бессонница", "апатия", "тоска", "радость", "ангедония"]`. Какое слово выбивается и почему?

**Интерпретация для психолога:** что означает семантическая близость слов в языке пациента? Если у человека слово *«мать»* близко к *«боль»* и *«страх»*, а не к *«тепло»* и *«забота»* — это может быть маркером травмы. Как вы можете использовать этот инструмент в практике?




In [ ]:
# Ваш код решения задачи:


### Задание 2. Анализ эмоций в тексте пациента

**Описание.** Используйте модель `ilyali034/rubert-emotion-ru-large` для анализа эмоционального содержания текста пациента. Модель классифицирует текст по 10 базовым эмоциям Изарда (радость, печаль, гнев, энтузиазм, удивление, отвращение, страх, вина, стыд, нейтральное).

**Текст для анализа:**

> *«Я больше не могу. Всё кажется бессмысленным. Ничего не радует. Просыпаюсь утром и не хочу вставать. Чувствую себя обузой для семьи. Мне стыдно, что я не могу справиться с собой.»*

**Требуется:**

1. Загрузите модель с Hugging Face с помощью `pipeline("text-classification", model="ilyali034/rubert-emotion-ru-large")`.

2. Проанализируйте текст пациента. Выведите все эмоции с вероятностями (только те, где вероятность > 0.1).

3. Напишите интерпретацию результатов:
   - Какие эмоции доминируют?
   - Что это говорит о состоянии пациента?
   - Какие эмоции могли бы быть маркерами депрессивного состояния?
   - Какие дополнительные вопросы вы бы задали пациенту после такого анализа?

4. *Дополнительно:* проанализируйте второй текст (придумайте или возьмите из открытого источника) и сравните эмоциональные профили двух пациентов.




In [ ]:
# Ваш код решения задачи:


### Задание 3. Поиск похожих клинических случаев

**Описание.** Используйте модель `sergeyzh/rubert-large-uncased-sts` (Sentence Transformer) для поиска семантически похожих клинических случаев. Это может помочь психологу находить в архиве случаи с похожей симптоматикой и сравнивать тактики терапии.

**Дано.** База из 10 анонимизированных клинических случаев (краткие описания симптомов):

```python
clinical_cases = [
    "Пациент жалуется на бессонницу, потерю аппетита и снижение настроения",
    "Больной отмечает тревогу, учащённое сердцебиение и чувство страха",
    "Пациент сообщает о чувстве вины, самообвинении и апатии",
    "Пациент чувствует себя хорошо, настроение стабильное, сон нормализовался",
    "Жалобы на головную боль, усталость и раздражительность",
    "Пациент говорит о панических атаках, страхе выходить из дома",
    "Больной сообщает о суицидальных мыслях, чувстве безнадёжности",
    "Пациент отмечает улучшение настроения, снижение тревоги после терапии",
    "Жалобы на проблемы с концентрацией внимания и памятью",
    "Пациент описывает чувство одиночества и социальной изоляции"
]
```

**Новый пациент:**
```
new_patient = "Пациент жалуется на тревогу, нарушение сна и чувство страха"
```

**Требуется:**

1. Загрузите модель `sergeyzh/rubert-large-uncased-sts`.

2. Получите эмбеддинги для всех клинических случаев и для нового пациента.

3. Вычислите косинусное сходство между новым пациентом и каждым случаем из базы.

4. Отсортируйте случаи по убыванию сходства. Выведите топ-3 наиболее похожих случая с коэффициентами сходства.

5. Напишите интерпретацию: что общего у найденных случаев? Какую информацию это может дать практикующему психологу?




In [ ]:
# Ваш код решения задачи:


### Задание 4. Извлечение симптомов и оценка суицидального риска

**Описание.** В этом задании вы используете модель `astromis/presuisidal_rubert` для оценки суицидального риска в тексте пациента, а также попробуете вручную извлечь симптомы (имитация работы модели).

**Текст пациента:**

> *«Пациент С., 34 года, обратился с жалобами на сниженное настроение, потерю интереса к работе и хобби. Отмечает трудности с засыпанием, ранние пробуждения, снижение аппетита. Сообщает о чувстве вины, самообвинении. На вопрос о суицидальных мыслях отвечает: "Иногда думаю, что лучше бы меня не было, но конкретных планов нет". В анамнезе — эпизоды депрессии 3 года назад. Алкоголь — эпизодически.»*

**Требуется:**

1. Загрузите модель `astromis/presuisidal_rubert` (или, если модель недоступна, используйте `astromis/presuisidal_rubert` — проверьте актуальное название на Hugging Face).

2. Проанализируйте текст на наличие суицидальных маркеров. Выведите результат классификации (например, "риск присутствует" или "риск не обнаружен") и вероятность.

3. **Вручную (имитируя работу модели):** выпишите все симптомы, упомянутые в тексте, и разделите их на категории:
   - **Эмоциональные** (сниженное настроение, чувство вины, самообвинение и т.д.)
   - **Соматические** (аппетит, сон, вес и т.д.)
   - **Поведенческие** (социальная активность, работа и т.д.)
   - **Когнитивные** (концентрация, память и т.д.)

4. Напишите краткое резюме (1 абзац): на основе выделенных симптомов, какое состояние вы бы предположили? Какие дополнительные вопросы вы бы задали пациенту?




In [ ]:
# Ваш код решения задачи:


### Задание 5. Генерация резюме терапевтической сессии (GPT-подход)

**Описание.** В этом задании вы познакомитесь с генеративной способностью LLM. Поскольку запуск больших моделей (например, Llama-3) требует значительных ресурсов, мы будем использовать простой подход: вы вручную составите резюме сессии, а затем сравните его с тем, как могла бы сделать модель (имитация).

**Задача.** У вас есть расшифровка фрагмента терапевтической сессии (диалог терапевта и пациента). Составьте **краткое резюме** (10–15 предложений), выделяя:
- основные жалобы пациента,
- ключевые симптомы,
- динамику состояния (если есть),
- терапевтические интервенции (что делал терапевт),
- план на следующую сессию.

**Диалог (сокращённый):**

> **Терапевт:** Здравствуйте. Как у вас дела на этой неделе?
>
> **Пациент:** Не очень. Опять не мог спать почти всю неделю. Засыпаю с трудом, просыпаюсь в 4 утра и не могу уснуть.
>
> **Терапевт:** Я понимаю, это очень утомляет. А что происходило днём? Удавалось ли заниматься делами?
>
> **Пациент:** Нет, я вообще ничего не делал. Лежал на диване, смотрел в потолок. Даже не хотелось есть. Мне кажется, что я никому не нужен, что я обуза для семьи.
>
> **Терапевт:** Я слышу, что вы чувствуете себя очень одиноко и обесцененно. Вы говорили с семьёй об этом?
>
> **Пациент:** Нет, я не хочу их тревожить. У них свои проблемы.
>
> **Терапевт:** Понимаю. Это защитная реакция — не хотеть быть обузой. Но ваши близкие, скорее всего, хотят быть рядом. Может быть, на следующей неделе попробуем вместе подумать, как вы можете мягко поговорить с ними?
>
> **Пациент:** Наверное... я попробую.

**Требования к резюме:**

- Используйте клиническую лексику (бессонница, сниженное настроение, ангедония, социальная изоляция).
- Укажите терапевтические интервенции (валидация чувств, предложение работать над коммуникацией).
- Сформулируйте план: обсуждение коммуникации с семьёй.

Напишите рефлексию (1 абзац): что было легко/сложно при составлении резюме? Где могла бы ошибиться GPT-модель?




```python
# Ваше резюме и рефлексия (текст, не код):
```

### Задание 6. Сравнение моделей: BERT vs GPT для психологических задач

**Описание.** В этом теоретико-практическом задании вы сравните два подхода — BERT (понимание) и GPT (генерация) — для двух клинических задач.

**Задача А. Классификация (BERT-подход)**

Даны 5 текстов пациентов. Классифицируйте их как «Депрессивный» или «Нейтральный».

Тексты для классификации:
1. «Сегодня хороший день, я гулял в парке и чувствовал себя спокойно.»
2. «Я устал от всего, ничего не хочется. Всё кажется серым и безрадостным.»
3. «Сон улучшился, стал просыпаться с меньшей тревогой.»
4. «Я не вижу смысла в жизни, зачем просыпаться утром?»
5. «Сходил к друзьям, было весело, но внутри всё равно пустота.»

Проведите классификацию **вручную** (имитация работы BERT). Для каждого текста укажите, почему вы отнесли его к той или иной категории (какие слова-маркеры вы использовали?).

**Задача Б. Генерация (GPT-подход)**

Для текста 4 («Я не вижу смысла в жизни, зачем просыпаться утром?») напишите, как бы мог ответить терапевтический GPT-бот (имитация). Какие фразы он мог бы использовать? Какие вопросы задать? Какие предостережения нужны?

**Задача В. Сравнение**

Напишите эссе (1 страница) на тему: *«Когда в психологии лучше использовать BERT, а когда — GPT?»* Опишите преимущества и ограничения каждого подхода. Приведите два конкретных примера из практики (можно гипотетических) для каждого подхода.



```python
# Ваш ответ (текст, не код):
```




## Часть 3. Этический анализ

**Задание 7. Этический протокол использования LLM в психологической практике**

**Описание.** Представьте, что вы — клинический психолог и руководитель центра психического здоровья. Вам предлагают внедрить систему на основе открытой LLM (например, `KhazarAI/MentalChat-16K` или `Horiznsky/Serenity-Llama-3.2-3B-Counsel`) для:
1. Автоматического анализа дневниковых записей пациентов (эмоциональный профиль, симптомы).
2. Генерации терапевтических рекомендаций между сессиями.

**Напишите этический протокол** (объёмом 2–3 страницы), в котором осветите:

1. **Информированное согласие** — что именно пациент должен знать об использовании ИИ? Как вы объясните ему ограничения модели (галлюцинации, возможность ошибки)?

2. **Конфиденциальность и безопасность** — как будут храниться данные? Будет ли использоваться облако или on-premise развёртывание? Кто имеет доступ к данным и результатам анализа?

3. **Человеческий контроль** — как организовать верификацию результатов модели? Кто отвечает за принятие финальных решений?

4. **Обработка ошибок и критических случаев** — что делать, если модель пропустит суицидальные маркеры? Что делать, если модель ложно сработает?

5. **Прозрачность и интерпретируемость** — как сделать работу модели понятной для пациента? Как вы будете объяснять, почему модель выдала тот или иной результат?

6. **Обучение персонала** — кто и как будет обучаться работе с системой?




```python
# Ваш ответ (текст, не код):
```


## Часть 4. Комплексное задание (повышенной сложности) — по желанию

**Задание 8. Fine-Tuning небольшой модели для классификации эмоций**

**Описание.** Если вы знакомы с основами машинного обучения и имеете доступ к GPU (или используете Google Colab), попробуйте выполнить дообучение (fine-tuning) модели `rubert-tiny` на небольшом датасете эмоций.

**Инструкция (упрощённая):**

1. Создайте синтетический датасет из 30–50 коротких фраз (по 10 фраз на 3–4 эмоции: печаль, радость, страх, нейтральное).

2. Загрузите модель `cointegrated/rubert-tiny`.

3. С помощью `Trainer` из библиотеки `transformers` выполните дообучение на 2–3 эпохи.

4. Оцените качество на тестовых примерах.

5. Напишите рефлексию (1 страница): что было сложно? Какие ограничения вы заметили? Какой объём данных нужен для дообучения в реальной клинической задаче?

Это задание требует предварительного знакомства с PyTorch и Hugging Face Trainer. Если вы не знакомы — пропустите.





In [ ]:
# Ваш код решения задачи:

## Критерии оценки

| Компонент | Процент | Описание |
|-----------|---------|----------|
| Теоретические вопросы (Часть 1) | 15% | Полнота и правильность ответов |
| Задание 1 (семантические соседи) | 10% | Корректность кода, интерпретация |
| Задание 2 (эмоциональный анализ) | 15% | Корректность кода, глубина интерпретации |
| Задание 3 (поиск похожих случаев) | 10% | Корректность кода, анализ результатов |
| Задание 4 (извлечение симптомов) | 15% | Полнота выделения симптомов, качество резюме |
| Задание 5 (генерация резюме) | 10% | Качество резюме, рефлексия |
| Задание 6 (сравнение BERT и GPT) | 10% | Глубина анализа, аргументированность |
| Задание 7 (этический протокол) | 15% | Полнота, практичность, аргументированность |
| Задание 8 (fine-tuning, бонус) | +5% | Корректность, рефлексия |

---

## Требования к сдаче

- Пришлите **один файл** (Jupyter Notebook `.ipynb` или Python `.py`) со всеми заданиями, кодом и текстовыми комментариями.
- Эссе и этический протокол (Задания 6 и 7) приложите в виде текстовых ячеек в Notebook или отдельного документа (`.txt` или `.pdf`).
- Убедитесь, что код выполняется без ошибок (укажите версии библиотек при необходимости).
- Все результаты анализа сопровождайте интерпретацией с точки зрения психолога.

---

## Заключение

Данная практическая работа проведёт вас через полный цикл работы с современными NLP-моделями — от базовых эмбеддингов до этических размышлений об использовании ИИ в клинической практике. Вы не только освоите инструменты, но и научитесь **критически оценивать** их применение в психологии.

**Главный вывод:** современные NLP-модели — это мощные помощники, но они не заменяют клиническое мышление. Ответственность за решения всегда остаётся за психологом.

---

**Срок выполнения: 2 недели.**